In [ ]:
import os, requests

GRAPHQL_URL = os.getenv("DATAHUB_GRAPHQL_URL", "http://datahub-gms:8080/api/graphql")
TOKEN = os.getenv("DATAHUB_TOKEN")
if not TOKEN:
    raise SystemExit("DATAHUB_TOKEN not set")

HEADERS = {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"}

def gql(query: str, variables: dict):
    r = requests.post(GRAPHQL_URL, json={"query": query, "variables": variables}, headers=HEADERS, timeout=60)
    r.raise_for_status()
    data = r.json()
    if data.get("errors"):
        raise RuntimeError(data["errors"])
    return data["data"]

# 1) Start from a known dataset (adjust if needed)
DATASET_URN = "urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD)"

GET_PARENTS = """
query getParents($urn: String!) {
  dataset(urn: $urn) {
    parentContainers {
      containers {
        urn
        properties { name description }
      }
    }
  }
}
"""
parents = gql(GET_PARENTS, {"urn": DATASET_URN})
containers = parents["dataset"]["parentContainers"]["containers"] if parents.get("dataset") else []
if not containers:
    raise SystemExit("No parent containers found. Your ingestion may not be creating database containers yet.")

print("Parent containers (closest first):")
for c in containers:
    print("-", c["properties"]["name"], "->", c["urn"])

# 2) Choose the DB container: prefer exact name 'classicmodels', else take the last (top-most)
db_container_urn = None
for c in containers:
    if (c["properties"].get("name") or "").lower() == "classicmodels":
        db_container_urn = c["urn"]; break
if not db_container_urn:
    db_container_urn = containers[-1]["urn"]

print("\nChosen DB container URN:", db_container_urn)

# 3) Update description (note: resourceUrn, not urn)
UPDATE_DESCRIPTION = """
mutation updateDescription($resourceUrn: String!, $description: String!) {
  updateDescription(input: { resourceUrn: $resourceUrn, description: $description })
}
"""

description_text = (
    "ClassicModels is a database that models an order-to-cash workflow for a classic car retailer. "
    "It includes customers, orders and orderdetails, payments, products and productlines, employees, and offices."
)

resp = gql(UPDATE_DESCRIPTION, {"resourceUrn": db_container_urn, "description": description_text})
print("\nupdateDescription result:", resp["updateDescription"], "✅")